In [ ]:
%run ../../../Latex_macros.ipynb

# Pricing American Options with Q-Learning
### Optimal stopping meets reinforcement learning

## Why American options are hard to price

- **European option**: payoff only realized at a fixed maturity $T$
    - closed-form (Black-Scholes) or plain Monte Carlo suffices
- **American option**: the holder may exercise at *any* time up to $T$
- Pricing it means solving for the **optimal exercise decision at every point in time**, not just averaging a fixed payoff
- That turns pricing into a **sequential decision problem**

## Optimal stopping, in words

- The option's value today = the value of following the *best* exercise strategy
- At any moment, compare:
  - payoff from exercising **now**, vs.
  - expected discounted value of **waiting** and deciding later
- "Compare now vs. continue optimally" is exactly the structure reinforcement learning solves for general sequential decisions
- That's why RL is a natural fit: this session applies the tools we just met (action values, Q-learning, exploration) to options

## Review: the MDP

*Covered in A0 and A1; skipped in the slideshow when this deck follows them.*

A **Markov Decision Process** describes an agent choosing actions over time:

- **State** $\state$ &mdash; everything relevant about the current situation
- **Action** $\act$ &mdash; a choice available in that state
- **Reward** $\rew$ &mdash; immediate payoff for taking $\act$ in $\state$
- **Transition** &mdash; the state evolves (possibly randomly) to a new state $\state'$
- **Discount factor** $\disc \in [0,1]$ &mdash; how much a future reward is worth today
- **Goal**: choose actions to maximize total discounted reward

## Review: the action value function

*Covered in A4; skipped in the slideshow when this deck follows it.*

$\actvalfun(\state, \act)$ = expected total discounted future reward if you take action $\act$ in state $\state$, then act optimally forever after

- Knowing $\actvalfun$ exactly gives the optimal action for free: $\argmax{\act} \actvalfun(\state, \act)$
- The optimal action value $\actvalfun_*$ satisfies the **Bellman optimality equation**:

$$\actvalfun_*(\state, \act) = \E\left[\rew + \disc \max{\act'} \actvalfun_*(\state', \act') \right]$$

"value of acting now = immediate reward + discounted value of behaving optimally from then on"

## Review: Q-learning

*Covered in A4; skipped in the slideshow when this deck follows it.*

- We rarely know the transition probabilities or reward function in closed form &mdash; but we *can* simulate or observe experience: tuples $(\state, \act, \rew, \state')$
- Q-learning nudges a table of estimates toward the Bellman equation using each observed transition:

$$\actvalfun(\state, \act) \leftarrow \actvalfun(\state, \act) + \alpha\Big[r + \disc\max{\act'}\actvalfun(\state', \act') - \actvalfun(\state, \act)\Big]$$

- $\alpha$: learning rate. The bracketed term is the **TD error** &mdash; how wrong the current estimate was
- Repeat over many simulated episodes and $\actvalfun$ converges toward the true optimal Q-function

## Review: exploration

*Covered in A4; skipped in the slideshow when this deck follows it.*

- Always taking the currently-best action means some actions in some states are never tried &mdash; their $\actvalfun$ estimate never improves, and might be wrong
- **$\epsilon$-greedy**: with probability $\epsilon$, take a random action instead of the greedy one; otherwise take $\argmax{\act} \actvalfun(\state, \act)$
- Common practice: start with a high $\epsilon$ (explore a lot early) and decay it toward 0 as training progresses

## Back to options: what's the state and action?

Map the general MDP onto "should I exercise this option?"

- **State** $\state = (t, j)$: the time step $t$ and the number $j$ of down-moves so far
    - the price is then $S_t = S_0 u^{\,t-j} d^{\,j}$ (the tree below explains why $j$ is enough)
    - $S_0$ is today's price
    - $u > 1$ and $d < 1$ are the factors for an up and a down move
- **Actions** $\act \in \{\text{exercise}, \text{continue}\}$

That's the whole mapping needed to make this a Q-learning problem

## Reward and discounting for options

- **Exercise now** (put with strike $K$): reward $= \text{max}(K - S_t, 0)$
    - the episode ends: no more decisions after exercising
- **Continue**: reward $= 0$ this step, move to the next node along the simulated price path
    - price up: $(t{+}1, j)$; price down: $(t{+}1, j{+}1)$ (the tree below shows why)
- At maturity $t = T$: exercise is **forced**
    - reward $= \text{max}(K - S_T, 0)$ whichever action was chosen
    - $T$ is terminal: its value is the payoff, so the table is learned only for $t < T$
    - *continue* at the last step before $T$ is worth the discounted payoff at $T$
- Discount factor $\disc = e^{-r\Delta t}$ &mdash; risk-free discounting *is* RL's discount factor here
    - $r$ is the risk-free rate, $\Delta t = T / n_{\text{steps}}$ the length of a step ($n_{\text{steps}}$ is the number of steps up to $T$)
- So $\actvalfun(\state, \text{continue})$ plays the role of the option's **continuation value**, and $\actvalfun(\state, \text{exercise})$ its **immediate exercise value**

## Tabular Q-learning needs a finite state space

> **A table has one entry per state, so the states must be finitely many: time and price can take only finitely many values.**

- $\actvalfun(\state, \act)$ as a lookup table requires finitely many states
- cut time into $n_{\text{steps}}$ equal intervals of length $\Delta t = T / n_{\text{steps}}$
- at each step the price moves **up** ($\times u$, with $u > 1$) or **down** ($\times d$, with $d < 1$)
- a first idea: let each sequence of moves be its own state

In [ ]:
# Purely structural -- no prices needed, just the shape of the tree.
# Every up/down sequence is its own node: 2**t nodes at step t.
# (The chart layout lives in lecture_viz.py: it is plot mechanics,
# not something about Q-learning or options.)
from lecture_viz import plot_naive_tree

plot_naive_tree(n=4)

- the red numbers count the nodes: $2^t$ at step $t$, so $16$ at $t = 4$
- with $n_{\text{steps}} = 50$ that is about $10^{15}$ nodes at the last step: far too many rows for a table
- "up, then down" and "down, then up" would be different rows, although only the order differs

## Only the number of down-moves matters: the paths recombine

> **Different orders of moves that reach the same price are the same state.**

- the price after $t$ steps depends only on $j$, the number of down-moves so far, $0 \le j \le t$
- so a state is the pair $(t, j)$, with exactly $t+1$ nodes at step $t$
- node $(t,j)$ has price $S_0 u^{\,t-j} d^{\,j}$

In [ ]:
# The same tree, with nodes that share a price merged (a recombining,
# Cox-Ross-Rubinstein tree): t+1 nodes at step t. Follow the two colored
# paths: different order of moves, same ending node.
from lecture_viz import plot_recombining_tree

plot_recombining_tree(n=4)

- from $(t,j)$ the price goes **up** to $(t{+}1,j)$ with probability $p$, or **down** to $(t{+}1,j{+}1)$ with probability $1-p$
    - $p$ is the risk-neutral probability: a pricing device, not a forecast
    - the assignment supplies $u$, $d$ and $p$
- recombining also **pools experience**: both histories have the same future, so visiting one teaches the table about the other
- without it you would have to visit and learn all $2^t$ historically distinct nodes instead of $t+1$

## The table: one cell per node

> **The tree is already a finite state space: the table has one cell per node $(t, j)$, holding two numbers.**

In [ ]:
# The Q table over the recombining tree: each node (t, j), j <= t, is one cell and holds
# two numbers: the value of continuing (C) and of exercising (E).
from lecture_viz import plot_q_table

plot_q_table(n=4)

- the two numbers in a cell: the value of continuing (C) and of exercising (E)
- cells learned: $n_{\text{steps}}(n_{\text{steps}}+1)/2$, about $1\,300$ for $n_{\text{steps}} = 50$, against about $10^{15}$ without recombining
- the last column ($t = n_{\text{steps}}$) is drawn but not learned: exercise is forced there


## The table: resolution, and what comes next

- no separate discretization step: the tree itself cuts time (into $n_{\text{steps}}$ steps) and price (onto the nodes $S_0 u^{\,t-j} d^{\,j}$)
- $n_{\text{steps}}$ is still a resolution choice
    - more steps: a finer approximation to continuous time
    - but more rows to learn from the same simulated experience: the usual bias and data tradeoff
- none of this is Q-learning yet: there is no update rule
    - it shows that "the tree already is a finite MDP"
    - the training loop then uses it

## The whole tree, with a few sample walks

- a preview of exploring starts (coming up): each colored path is one simulated episode
- the start is not always the root: an episode may begin at any node
- synthetic numbers, not the assignment's option; no Q-learning runs here

In [ ]:
# Illustrative only -- synthetic parameters, not the assignment's option.
# This cell does NOT run Q-learning: it draws the lattice (the layout code
# is in lecture_viz.py) and overlays three hand-written example walks.
# Each walk is (start time, start j, moves): the start is NOT always the
# root, and a move is U (price up, j unchanged) or D (price down, j + 1).
from lecture_viz import plot_lattice_with_walks

walks = [(1, 0, 'UDUUDUU'), (3, 3, 'DUDDU'), (5, 2, 'UDD')]
plot_lattice_with_walks(walks)

## Aside (optional): recombining is a modeling choice, not automatic

- Suppose the stock paid a **discrete dollar dividend** $D$ on some date. The tempting quick fix is to just subtract $D$ from the stock price at that time step. Don't &mdash; different nodes at that step have *different* prices, so $S - D$ isn't the same multiplicative adjustment at each one, and up/down moves from different pre-dividend nodes stop landing on a shared future grid. The tree stops recombining, and you're back to something closer to $2^t$ states
- Notice *where* this mistake would live: in the **environment's dynamics** (how price evolves), not the reward. It's a good example of how an environment-design choice that looks purely economic ("how do I model a dividend?") can silently destroy a structural property (recombining) that everything else here quietly depends on
- The standard fixes (a continuous/proportional dividend yield, or the "escrowed dividend" trick for discrete dividends) exist specifically to preserve recombining &mdash; not pursued further here, but worth knowing they exist before reaching for the naive subtraction

## Choosing the learning rate $\alpha$

> **Use a small constant $\alpha$: small enough to average over noisy samples, constant enough to keep adapting as the bootstrapped targets improve.**

- classical theory says a **decaying** $\alpha$, e.g. $\alpha = 1/N(\state, \act)$, guarantees convergence
    - it is the same idea as a running sample average
- but Q-learning **bootstraps**: the target for one state uses the *current* estimate of another state, itself still being learned
    - a state visited early and often can shrink its $\alpha$ to near zero *before* the states it bootstraps from have converged
    - it then freezes at a stale estimate it can never correct
- the practical fix: a small **constant** $\alpha$ tolerates a moving target, at the cost of never fully settling

**The two extremes of $\alpha$ are both broken**

- $\alpha = 0$ is not a gentler version of the fix
    - it discards every update, so nothing is ever learned
- $\alpha$ near $1$ makes $\actvalfun(\state, \act) \leftarrow \text{target}$
    - each update *replaces* the estimate instead of blending with it
    - the table chases the noise of whichever single sample arrived last
    - with bootstrapping, that noise can propagate from state to state instead of averaging out
- earlier sessions used a large $\alpha = 0.9$ safely
    - but only because that Frozen Lake was deterministic: nothing random to chase

## Exploring starts: covering the whole state space

> **Exploring starts begin episodes in states drawn broadly across the space, so every state gets direct experience.**

- recall from A4: $\epsilon$-greedy chooses *actions*; exploring starts choose where episodes *start*
- our version covers states only, not state-action pairs
- it matters most where bootstrapping is involved (the $\alpha$ slides)
    - a state's update uses the *current* estimate at the next state
    - if that state is undertrained, its error flows into every earlier state that bootstraps off it

**Why Frozen Lake did not need exploring starts, and the tree does**

- Frozen Lake: the state is the grid position, with no time index
    - the same cell can be revisited endlessly: a **recurrent** environment (states can be revisited)
    - so acting $\epsilon$-greedily keeps returning to every cell
- our tree: time is *part of the state*, and every transition advances $t$
    - the tree has no loops (a *DAG*): an episode that passes step $t$ never revisits it
    - each episode gives one sample of where you land at each $t$
    - that placement is fixed by the binomial math of the walk, not by how long you train
    - a corner the natural dynamics rarely reach stays rarely reached, however many episodes you run
- so you must deliberately *start* there

## Training loop, at a glance

> **Many simulated episodes, one Q-learning update after every step, then read the policy off the table.**

1. generate a large number of training episodes with **exploring starts**
    - so every $(t, j)$ node gets real coverage
    - not just the nodes near the option's actual starting price
2. for each episode, step forward through time until the agent exercises or reaches $T$
    - pick an action $\epsilon$-greedily
    - observe the resulting $(\state, \act, \rew, \state')$
3. apply the Q-learning update after every step, with a small **constant** $\alpha$
4. after training, read off the policy $\argmax{\act} \actvalfun(\state, \act)$ in every state
    - this **is** the estimated optimal exercise strategy

**"A large number" means $N_{\text{episodes}} = 1{,}000{,}000$**

- that can look absurd for a tree with only a few hundred states
- with a small constant $\alpha$, each state needs on the order of hundreds of updates before its estimate settles
- exploring starts must spread that experience thinly enough to still reach the least-favored state
- so six or seven figures of episodes is normal for tabular Q-learning, not a sign of a bug
- curious where the number comes from? See the optional deep dive in the last cells of the notebook (not shown in class)

## Reading the answer: price and exercise boundary

> **The price is the value of the root; the policy at every node says exercise or continue.**

- the option's value now is $\max{\act} \actvalfun(0, 0, \act)$
- the policy is $\argmax{\act} \actvalfun(t, j, \act)$
- the **exercise boundary** at step $t$ is the *highest price* at which the policy says exercise
    - below it exercising pays
    - above it waiting is worth more
- careful with direction: a larger $j$ means a *lower* price
    - so the boundary sits at the **smallest** $j$ whose policy says exercise
- plotted against $t$ the boundary is a curve, and the assignment compares yours with the exact one

## How do we know it worked?

> **Q-learning trains on the *same* tree the pricing method uses, so the benchmark is the exact solution to the same MDP.**

- the two methods get there differently
    - the **tree pricer** computes its answer once, by exact backward induction
        - start at maturity, where values are known
        - work back one step at a time
    - **Q-learning** reaches its estimate by sampling and learning: many simulated episodes, each nudging the table with a TD update
- the check is still meaningful
    - it separates "is my Q-learning implementation correct" from "did I discretize the problem reasonably"
    - there is only one discretization now
- for continuous-price models with no exact tree solution, a standard alternative is **Longstaff-Schwartz**
    - it estimates continuation values by regression
    - it is not needed here

## DP vs. Q-learning: two ways to solve the same tree

> **DP needs the model and sweeps once; Q-learning needs only samples and sweeps many times.**

- `binomial_tree_american_put` is classical **dynamic programming**: one backward sweep from maturity
    - each node is computed **exactly once**
    - it depends only on next-step values that are already final
    - earlier we swept repeatedly because states depended on each other in loops
    - a tree has no loops and is ordered by time
- that sweep requires *knowing the model*
    - an exact expectation at each node needs $p$ and the payoff function up front


**What Q-learning gives up, and what it gains**

- Q-learning never uses $p$ directly
    - it only needs to **sample** a transition, or observe one in a real market
    - that is what **model-free** means, and it is the actual point of this exercise
- what does not carry over: Q-learning sees only sampled transitions, so it gets no backward ordering to exploit
    - the assignment's last question asks what that costs, using the numbers for this tree
- general principle, not specific to options
    - **DP dominates whenever the model is known**
    - **RL earns its keep exactly when it isn't**: real markets give observed price histories, not a risk-neutral probability

## Recap

- American option pricing = optimal stopping = a sequential "decide now or later" problem
- state $(t, j)$ on a recombining binomial tree
    - actions $\{\text{exercise}, \text{continue}\}$
    - reward = payoff on exercise, and $\disc = e^{-r\Delta t}$
- Q-learning learns $\actvalfun(\state, \act)$ from simulated tree walks
    - TD updates with a small constant $\alpha$
    - $\epsilon$-greedy action exploration
    - exploring starts for state coverage
- validate against the tree's own exact backward-induction price and boundary
    - the assignment builds and checks exactly this
- DP solves the same tree exactly in one model-based sweep
    - Q-learning trades away the known-model requirement for sampled, model-free convergence
    - that trade is the whole reason Q-learning matters for real markets

## Deep dive (optional): why one million episodes?

*Skipped in a live presentation &mdash; read this on your own if you want to know where `N_EPISODES = 1,000,000` actually came from. It's not an arbitrary round number, and the reasoning generalizes past this one assignment.*

## Step 1: how hard is the hardest state to reach?

- suppose the start time $t_0$ is drawn uniformly from $\{0,\dots,n_{\text{steps}}-1\}$, then $j_0$ uniformly from $\{0,\dots,t_0\}$
- So $P(t_0=t,\, j_0=j) = \dfrac{1}{n_{\text{steps}}(t+1)}$ &mdash; states at large $t$ split their layer's probability among more competitors, so they're individually *harder* to hit as a start than states at small $t$
- The single hardest state: $t_0 = n_{\text{steps}}-1$, where $n_{\text{steps}}$ values of $j$ split $1/n_{\text{steps}}$ of the probability, giving each one $p_{\min} = 1/n_{\text{steps}}^2$
- Over $N$ episodes, the *expected* number of times that worst-case state is chosen as a start is $N \cdot p_{\min} = N / n_{\text{steps}}^2$ &mdash; a real, computable lower bound on its total visits (anything it also picks up mid-episode only adds to this)

## Step 2: how many visits does a state actually need?

- With a **constant** learning rate, a state's estimate forgets its initial value geometrically: after $k$ updates against a roughly stable target, the leftover bias shrinks like $(1-\alpha)^k$
- Getting that comfortably small takes roughly $k \approx 3/\alpha$ updates &mdash; about 300 for $\alpha = 0.01$
- So a reasonable target: every state should be visited (as a start, at minimum) on the order of $m \approx 1/\alpha$ times
- Combining with Step 1: $N \gtrsim \dfrac{C}{\alpha}\, n_{\text{steps}}^2$ for some small constant $C$

In [ ]:
# Plugging in the assignment's own numbers.
alpha = 0.01
n_steps = 25

base_estimate = n_steps ** 2 / alpha
actual = 1_000_000

print(f'n_steps^2 / alpha  = {base_estimate:,.0f}')
print(f'Actual N_EPISODES  = {actual:,}  (~{actual / base_estimate:.0f}x margin)')

## The honest caveat

- This estimate only guarantees each state gets *some* direct experience as a start &mdash; it says nothing about **bootstrapping**: a state can't really settle until its *successor* states already have decent estimates, and information has to propagate backward through up to $n_{\text{steps}}$ layers of the tree
- A plausible (not proven here) extra factor of $n_{\text{steps}}$ for that propagation would put the estimate around $n_{\text{steps}}^3/\alpha \approx 1.56$ million &mdash; suspiciously close to the actual `N_EPISODES`, worth noticing without over-claiming it as derived
- A fully rigorous version of this exists in the RL theory literature as **sample-complexity bounds for tabular Q-learning**. This back-of-envelope version is meant as a sanity check and a design habit, not a substitute for that theory
- Takeaway: $1{,}000{,}000$ isn't a magic number &mdash; it's in the right *ballpark* this kind of reasoning would suggest, with a comfortable safety margin on top

## The bigger lesson

A hyperparameter that looks arbitrary usually isn't &mdash; it's just arbitrary *to someone who hasn't asked what it's fighting against*. The general pattern, useful well beyond this assignment:

1. **Identify the bottleneck** &mdash; here, the least-visited state under your own sampling scheme
2. **Write down a probability or convergence argument** for it &mdash; here, the exploring-start hit probability and the constant-$\alpha$ forgetting rate
3. **Compute a number** from that argument, even a rough one
4. **Sanity-check it against whatever value you're actually using** &mdash; a huge mismatch either way is worth understanding, not shrugging off

This beats pure trial-and-error tuning even when the argument is informal: it tells you *why* a value needs to be roughly what it is, not just *that* it happens to work &mdash; which is exactly what lets you predict how it should change if $n_{\text{steps}}$ or $\alpha$ changes, instead of re-tuning from scratch.